# Project 1: Word Embeddings / Recurrent Neural Networks

In project is part of the NLP module held in the spring of 2026. It covers two different training architectures:
- Word embeddings (word2vec, GloVe, or fastText) together with a classifier (2-layer with ReLU
non-linearity)
- A 2-layer RNN architecture (LSTM or GRU, use the PyTorch implementations), and a two-layer
classifier with a ReLU.

## Introduction

#### todo
- words and biases link
- sources
- tools

## Setup

In [37]:
# pip install datasets wandb fasttext torch nltk

In [38]:
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
import numpy as np
import re
import string
import ssl
import wandb
import nltk
import torch
import torch.nn as nn
import fasttext.util

In [39]:
wandb.login()

True

## Preprocessing

### Load Dataset

In [40]:
# Load dataset directly form parquet files because dataset scripts are no longer supported
train_split = load_dataset("ybisk/piqa", split="train[:-1000]", revision='refs/convert/parquet')
valid_split = load_dataset("ybisk/piqa", split="train[-1000:]", revision='refs/convert/parquet')
test_split = load_dataset("ybisk/piqa", split="validation", revision='refs/convert/parquet')

### Explore Dataset

In [41]:
print(f"Dataset split sizes")
print(f"Train: {len(train_split)}\nValidation: {len(valid_split)}\nTest: {len(test_split)}")

print("__________________________________________________________")
print(f"Example Row: {train_split[0]}")

print("__________________________________________________________")
COL_GOAL = 'goal'
COL_SOL1 = 'sol1'
COL_SOL2 = 'sol2'
COL_LABEL = 'label'
print(f"Features: {train_split.features}\nSelected: {[COL_GOAL, COL_SOL1, COL_SOL2]}\nTarget: {COL_LABEL}")
print("__________________________________________________________")
print(f"Number of rows labeled with class '0' in train: {train_split[COL_LABEL].count(0)}")
print(f"Number of rows labeled with class '0' in total: {train_split[COL_LABEL].count(0) + valid_split[COL_LABEL].count(0) + test_split[COL_LABEL].count(0)}\n")
print(f"Number of rows labeled with class '1' in train: {train_split[COL_LABEL].count(1)}")
print(f"Number of rows labeled with class '1' in total: {train_split[COL_LABEL].count(1) + valid_split[COL_LABEL].count(1) + test_split[COL_LABEL].count(1)}")

# regex source: https://apxml.com/courses/nlp-fundamentals/chapter-1-nlp-text-processing-techniques/handling-text-noise
# verified with: https://regex101.com
regex_pattern = re.compile(r'<[^>]+>', re.IGNORECASE)
html_elements = 0

for split in [train_split, valid_split, test_split]:
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_GOAL])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL1])))
    html_elements += len(split.filter(lambda row: regex_pattern.search(row[COL_SOL2])))

print("__________________________________________________________")
print(f"Number of HTML elements found: {html_elements}")



Dataset split sizes
Train: 15113
Validation: 1000
Test: 1838
__________________________________________________________
Example Row: {'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1}
__________________________________________________________
Features: {'goal': Value('string'), 'sol1': Value('string'), 'sol2': Value('string'), 'label': ClassLabel(names=['0', '1'])}
Selected: ['goal', 'sol1', 'sol2']
Target: label
__________________________________________________________
Number of rows labeled with class '0' in train: 7536
Number of rows labeled with class '0' in total: 8963

Number of rows labeled with class '1' in train: 7577
Number of rows labeled with class '1' in total: 8988
__________________________________________________________
Number of HTML elements found: 0


### Tokenization / Punctuation Removal / Input

Stemming or Lemmatization only has to be done when the embedding model is trained with such text. 
It is not done because the FastText model is trained on raw words. With Stemming or Lemmatization those vectors would be lost.

Stopword removal would not help increasing the model performance.
Punctuation removal is done. 

In [42]:
# Skip HTTPS certificate verification to allow download
ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('punkt')
nltk.download('punkt_tab')

# Separation token is used for optimizing the hidden state of RNN
SEPARATION_TOKEN = '<SEP>'

INPUT1_FIELD = 'input1'
INPUT2_FIELD = 'input2'

# Found by the analysis of processed training set further down. 
TRUNCATION_LENGTH = 62 

def punctuation_removal(tokens):
    return [token for token in tokens if token not in string.punctuation]

def truncate(tokens):
    return tokens[:TRUNCATION_LENGTH]

def preprocess(text):
    # By lowercasing the text the amount of tokens is reduced without loosing information.
    text_lower = text.lower()
    tokens = nltk.word_tokenize(text_lower)
    tokens = punctuation_removal(tokens)
    return tokens

def format_input(goal, sol):
    return goal + [SEPARATION_TOKEN] + sol

def preprocess_row(row):
    preprocessed_goal = preprocess(row[COL_GOAL])
    preprocessed_sol1 = preprocess(row[COL_SOL1])
    preprocessed_sol2 = preprocess(row[COL_SOL2])
    
    # By formatting the goal and solutions into two separate input fields we ensure that the model predicts independently.
    # What we can think about is processing the two solutions together.
    input1 = format_input(preprocessed_goal, preprocessed_sol1)
    input2 = format_input(preprocessed_goal, preprocessed_sol2)
    
    return {
        INPUT1_FIELD: truncate(input1),
        INPUT2_FIELD: truncate(input2)
    }

train_processed = train_split.map(preprocess_row)
valid_processed = valid_split.map(preprocess_row)
test_processed = test_split.map(preprocess_row)

print(f"Row before preprocessing: {train_split[0]}")
print(f"Processed row: {train_processed[0]}")

Row before preprocessing: {'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1}
Processed row: {'goal': "When boiling butter, when it's ready, you can", 'sol1': 'Pour it onto a plate', 'sol2': 'Pour it into a jar', 'label': 1, 'input1': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<SEP>', 'pour', 'it', 'onto', 'a', 'plate'], 'input2': ['when', 'boiling', 'butter', 'when', 'it', "'s", 'ready', 'you', 'can', '<SEP>', 'pour', 'it', 'into', 'a', 'jar']}


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/sachavogel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sachavogel/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


#### Truncation

In [43]:
lengths = [len(row[INPUT1_FIELD]) for row in train_processed] + [len(row[INPUT2_FIELD]) for row in train_processed]

print(f"Min length:    {np.min(lengths)}")
print(f"Max length:    {np.max(lengths)}")
print(f"Mean length:   {np.mean(lengths):.1f}")
print(f"Median length: {np.median(lengths):.1f}")
print(f"95th percentile: {np.percentile(lengths, 95):.1f}")
print(f"99th percentile: {np.percentile(lengths, 99):.1f}")

# Output without truncated training data
# Min length:    4
# Max length:    385
# Mean length:   27.0
# Median length: 22.0
# 95th percentile: 62.0
# 99th percentile: 97.0

# Those outputs tell us we can implement truncate logic into the processing step. 
# We choose to truncate everything in over the 95th percentile, that is also safe with fast text.

# We can check if we lose some solutions entirely. That would be the case if the goal itself matches or exceeds the length of 62.0.
goal_lengths = [len(row[COL_GOAL]) for row in train_processed]
print("__________________________________________________________")
print(f"Min goal length:    {np.min(goal_lengths)}")
print(f"Max goal length:    {np.max(goal_lengths)}")
print(f"Mean goal length:   {np.mean(goal_lengths):.1f}")
print(f"Median goal length: {np.median(goal_lengths):.1f}")
print(f"95th percentile: {np.percentile(goal_lengths, 95):.1f}")
print(f"99th percentile: {np.percentile(goal_lengths, 99):.1f}")
# todo handle truncation of goals because we have some processed input fields which do not have a solution. 

Min length:    4
Max length:    62
Mean length:   25.8
Median length: 22.0
95th percentile: 62.0
99th percentile: 62.0
__________________________________________________________
Min goal length:    2
Max goal length:    154
Mean goal length:   35.8
Median goal length: 34.0
95th percentile: 68.0
99th percentile: 86.9


### Vocabulary

In [44]:
UNK_TOKEN = '<UNK>'
PAD_TOKEN = '<PAD>'

def create_vocabulary(rows):
    v = set([SEPARATION_TOKEN, UNK_TOKEN, PAD_TOKEN])
    for row in rows:
        for token in row['input1'] + row['input2']:
            v.add(token)
    return v

def create_word2idx(vocabulary):
    w2i = {}
    for i, word in enumerate(vocabulary):
        w2i[word] = i
    return w2i

vocabulary = create_vocabulary(train_processed)
word2idx = create_word2idx(vocabulary)

print(f"Vocabulary size: {len(vocabulary)}")
print(f"Example vocabulary: {list(vocabulary)[:5]}")
print(f"Word to index for first word: {word2idx[list(vocabulary)[0]]}")

Vocabulary size: 15470
Example vocabulary: ['non-conducting', 'medjool', 'lasting', 'linger', 'wite-out']
Word to index for first word: 0


In [45]:
def check_unknown(tokens):
    return [token if token in vocabulary else UNK_TOKEN for token in tokens]
        
def replace_unknowns(row):
    return {
        INPUT1_FIELD: check_unknown(row[INPUT1_FIELD]),
        INPUT2_FIELD: check_unknown(row[INPUT2_FIELD])
    }

def count_unknowns(split):
    count = sum(token == UNK_TOKEN for row in split for token in row[INPUT1_FIELD] + row[INPUT2_FIELD])
    total = sum(len(row[INPUT1_FIELD]) + len(row[INPUT2_FIELD]) for row in split)
    return count, total

# Vocabulary is built from training set, therefore it does not have any unknown words.
valid_processed = valid_processed.map(replace_unknowns)
test_processed = test_processed.map(replace_unknowns)

unk_count, total_count = count_unknowns(train_processed)
print(f"{UNK_TOKEN} count in training set: {unk_count} ({unk_count * 100 / total_count:.2f}%)")
unk_count, total_count = count_unknowns(valid_processed)
print(f"{UNK_TOKEN} count in validation set: {unk_count} ({unk_count * 100 / total_count:.2f}%)")
unk_count, total_count = count_unknowns(test_processed)
print(f"{UNK_TOKEN} count in test set: {unk_count} ({unk_count * 100 / total_count:.2f}%)")

<UNK> count in training set: 0 (0.00%)
<UNK> count in validation set: 759 (1.43%)
<UNK> count in test set: 1516 (1.60%)


In [46]:
def tokens_to_ids(tokens):
    return [word2idx.get(token, word2idx[UNK_TOKEN]) for token in tokens]

def encode_row(row):
    return {
        INPUT1_FIELD: tokens_to_ids(row[INPUT1_FIELD]),
        INPUT2_FIELD: tokens_to_ids(row[INPUT2_FIELD]),
        COL_LABEL: row[COL_LABEL]
    }

def pad_sequence(seq):
    if len(seq) < TRUNCATION_LENGTH:
        return seq + [word2idx[PAD_TOKEN]] * (TRUNCATION_LENGTH - len(seq))
    return seq[:TRUNCATION_LENGTH]

def pad_row(row):
    return {
        INPUT1_FIELD: pad_sequence(row[INPUT1_FIELD]),
        INPUT2_FIELD: pad_sequence(row[INPUT2_FIELD]),
        COL_LABEL: row[COL_LABEL]
    }

train_encoded = train_processed.map(encode_row)
valid_encoded = valid_processed.map(encode_row)
test_encoded = test_processed.map(encode_row)
print(f"Example of encoded training data: {train_encoded[:5][INPUT1_FIELD]}")

train_padded = train_encoded.map(pad_row)
valid_padded = valid_encoded.map(pad_row)
test_padded = test_encoded.map(pad_row)
print("__________________________________________________________")
print(f"Example of padded training data: {train_padded[:5][INPUT1_FIELD]}")
print("__________________________________________________________")
print(f"Index of PAD_TOKEN: {tokens_to_ids([PAD_TOKEN])}")

Example of encoded training data: [[12255, 5457, 6291, 12255, 1342, 8953, 14917, 9866, 10902, 4059, 3938, 1342, 7564, 10315, 8704], [6033, 9021, 6461, 8038, 11537, 6033, 10315, 1257, 9866, 10902, 4059, 4086, 12994, 8038, 11392, 6033, 360, 1342, 6033, 2863, 13213, 8608, 13231], [1732, 14224, 9866, 11646, 8880, 4059, 4821, 10315, 1566, 6885, 10524, 12994, 1138], [1732, 14224, 9866, 14879, 8880, 4059, 1864, 1342, 12614, 7542, 6137, 7542, 1988, 6033, 1988, 10176], [9019, 1130, 4059, 3938, 2161, 15103, 10668, 2678, 10936, 7418, 4941, 1782, 10114, 6033, 9019, 7874, 13837, 7542, 6158, 20]]
__________________________________________________________
Example of padded training data: [[12255, 5457, 6291, 12255, 1342, 8953, 14917, 9866, 10902, 4059, 3938, 1342, 7564, 10315, 8704, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019, 6019,

### Batching

In [47]:
# Batch size chosen high because of suggestions in the course. 
BATCH_SIZE = 128

class PiqaDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset
        
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        row = self.dataset[idx]
        return (
            row[INPUT1_FIELD],
            row[INPUT2_FIELD],
            row[COL_LABEL]
        )

def to_tensor(row):
    return {
        INPUT1_FIELD: torch.tensor(row[INPUT1_FIELD], dtype=torch.long),
        INPUT2_FIELD: torch.tensor(row[INPUT2_FIELD], dtype=torch.long),
        COL_LABEL: torch.tensor(row[COL_LABEL], dtype=torch.long)
    }

train_tensor = train_padded.map(to_tensor)
valid_tensor = valid_padded.map(to_tensor)
test_tensor = test_padded.map(to_tensor)

# collate fn what is that?
# The training set has to be shuffled to ensure random order in training which makes training more stable.  
train_loader = DataLoader(PiqaDataset(train_tensor), batch_size=BATCH_SIZE, shuffle=True)
# Test and Validation should not be shuffled to ensure reproducibility and consistency of the model.
valid_loader = DataLoader(PiqaDataset(valid_tensor), batch_size=BATCH_SIZE)
test_loader = DataLoader(PiqaDataset(test_tensor), batch_size=BATCH_SIZE)

## Model

This project looks into the fastText model. 
The model handles bags as n-grams of characters, therefore it can handle unknown words great.
This results in higher inference times, but that is negligible on this PIQA task.
# todo look into subword model argumentation
# Potential challenge: How do you handle <UNK> in pretrained embeddings? This is a real issue you'll need to address in your model section. Pretrained embeddings don't have a vector for <UNK> — you'll need to either use a random vector, use the average of all embedding vectors, or use fastText's subword model (which can handle unknown words through character n-grams). fastText is actually the strongest choice here precisely because of this.
# Potential challenge: How do you handle <SEP> and <PAD> in pretrained embeddings? Same issue — these are custom tokens with no pretrained vector. You'll need to either initialize them randomly and freeze them, or learn them during training.

In [48]:
# We only want to download the model once
# fasttext.util.download_model('en', if_exists='ignore')

### Build Embedding Matrix

In [49]:
# This is how the 'cc.en.300.bin' model was trained
EMBEDDING_DIM = 300 

ft = fasttext.load_model('cc.en.300.bin')

def create_embedding_matrix():
    matrix = np.zeros((len(word2idx), EMBEDDING_DIM))
    
    for word, i in word2idx.items():
        if word == PAD_TOKEN:
            matrix[i] = np.zeros(EMBEDDING_DIM)
        elif word in [UNK_TOKEN, SEPARATION_TOKEN]:
            matrix[i] = np.random.normal(0, 0.1, EMBEDDING_DIM)
        else:
            matrix[i] = ft.get_word_vector(word)
    
    return torch.tensor(matrix, dtype=torch.float)
    
embedding_matrix = create_embedding_matrix()

print(f"Length of vocabulary: {len(vocabulary)}")
print(f"Embedding dimension: {EMBEDDING_DIM}")
print(f"Shape of the embedding matrix: {embedding_matrix.shape}")

Length of vocabulary: 15470
Embedding dimension: 300
Shape of the embedding matrix: torch.Size([15470, 300])


### Build Embedding Layer

In [54]:
embedding_layer = nn.Embedding(
    num_embeddings=embedding_matrix.shape[0], # That represents the vocabulary size
    embedding_dim=embedding_matrix.shape[1], # This is the embedding dimension
    padding_idx=word2idx[PAD_TOKEN] # We set the padding index to ensure that this vector does not influence model weights.
)

embedding_layer.weight = nn.Parameter(embedding_matrix, requires_grad=False) # We import and freeze the pretrained weights.

test_word = "butter"
test_index = torch.tensor([word2idx.get(test_word, word2idx[UNK_TOKEN])])
result = embedding_layer(test_index)
print(f"The word '{test_word}' has the vector shape: {result.shape}")

The word 'butter' has the vector shape: torch.Size([1, 300])


## Training

In [51]:
# todo

## Evaluation

In [52]:
# todo

## Interpretation

In [53]:
# todo